In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PySparkPractice") \
    .master("local[*]") \
    .getOrCreate()

print(spark.version)

C:\Users\dheer\anaconda3\envs\spark311\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


4.2.0


In [2]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [3]:
product_data = [
    (1, "iphone",  "01-01-2023", 1500000),
    (2, "samsung", "01-01-2023", 1100000),
    (3, "oneplus", "01-01-2023", 1100000),

    (1, "iphone",  "01-02-2023", 1300000),
    (2, "samsung", "01-02-2023", 1120000),
    (3, "oneplus", "01-02-2023", 1120000),

    (1, "iphone",  "01-03-2023", 1600000),
    (2, "samsung", "01-03-2023", 1080000),
    (3, "oneplus", "01-03-2023", 1160000),

    (1, "iphone",  "01-04-2023", 1700000),
    (2, "samsung", "01-04-2023", 1800000),
    (3, "oneplus", "01-04-2023", 1170000),

    (1, "iphone",  "01-05-2023", 1200000),
    (2, "samsung", "01-05-2023", 980000),
    (3, "oneplus", "01-05-2023", 1175000),

    (1, "iphone",  "01-06-2023", 1100000),
    (2, "samsung", "01-06-2023", 1100000),
    (3, "oneplus", "01-06-2023", 1200000)
]

schema = ["product_id", "product_name", "sales_date", "sales"]

product_df = spark.createDataFrame(product_data, schema)

product_df.show()

+----------+------------+----------+-------+
|product_id|product_name|sales_date|  sales|
+----------+------------+----------+-------+
|         1|      iphone|01-01-2023|1500000|
|         2|     samsung|01-01-2023|1100000|
|         3|     oneplus|01-01-2023|1100000|
|         1|      iphone|01-02-2023|1300000|
|         2|     samsung|01-02-2023|1120000|
|         3|     oneplus|01-02-2023|1120000|
|         1|      iphone|01-03-2023|1600000|
|         2|     samsung|01-03-2023|1080000|
|         3|     oneplus|01-03-2023|1160000|
|         1|      iphone|01-04-2023|1700000|
|         2|     samsung|01-04-2023|1800000|
|         3|     oneplus|01-04-2023|1170000|
|         1|      iphone|01-05-2023|1200000|
|         2|     samsung|01-05-2023| 980000|
|         3|     oneplus|01-05-2023|1175000|
|         1|      iphone|01-06-2023|1100000|
|         2|     samsung|01-06-2023|1100000|
|         3|     oneplus|01-06-2023|1200000|
+----------+------------+----------+-------+



In [10]:
window = Window.partitionBy('product_id').orderBy('sales_date')
last_month_df = product_df.withColumn('previous_month_sales', lag(col('sales'),1).over(window))
last_month_df.show()

+----------+------------+----------+-------+--------------------+
|product_id|product_name|sales_date|  sales|previous_month_sales|
+----------+------------+----------+-------+--------------------+
|         1|      iphone|01-01-2023|1500000|                NULL|
|         1|      iphone|01-02-2023|1300000|             1500000|
|         1|      iphone|01-03-2023|1600000|             1300000|
|         1|      iphone|01-04-2023|1700000|             1600000|
|         1|      iphone|01-05-2023|1200000|             1700000|
|         1|      iphone|01-06-2023|1100000|             1200000|
|         2|     samsung|01-01-2023|1100000|                NULL|
|         2|     samsung|01-02-2023|1120000|             1100000|
|         2|     samsung|01-03-2023|1080000|             1120000|
|         2|     samsung|01-04-2023|1800000|             1080000|
|         2|     samsung|01-05-2023| 980000|             1800000|
|         2|     samsung|01-06-2023|1100000|              980000|
|         

In [14]:
last_month_df.withColumn('per_loss_gain',
                        round(((col('sales')-col('previous_month_sales'))/col('sales'))*100,2))\
                        .show()

+----------+------------+----------+-------+--------------------+-------------+
|product_id|product_name|sales_date|  sales|previous_month_sales|per_loss_gain|
+----------+------------+----------+-------+--------------------+-------------+
|         1|      iphone|01-01-2023|1500000|                NULL|         NULL|
|         1|      iphone|01-02-2023|1300000|             1500000|       -15.38|
|         1|      iphone|01-03-2023|1600000|             1300000|        18.75|
|         1|      iphone|01-04-2023|1700000|             1600000|         5.88|
|         1|      iphone|01-05-2023|1200000|             1700000|       -41.67|
|         1|      iphone|01-06-2023|1100000|             1200000|        -9.09|
|         2|     samsung|01-01-2023|1100000|                NULL|         NULL|
|         2|     samsung|01-02-2023|1120000|             1100000|         1.79|
|         2|     samsung|01-03-2023|1080000|             1120000|         -3.7|
|         2|     samsung|01-04-2023|1800